<a href="https://colab.research.google.com/github/ssk-algoverse/sae-binding/blob/main/circuit/Analyse-Circuit-Sebastian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformer_lens

In [67]:
import torch
from huggingface_hub import hf_hub_download
from transformer_lens import HookedTransformer, HookedTransformerConfig, utils
import numpy as np
import pandas as pd
import ast
from torch.utils.data import Dataset, DataLoader

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "sojup/entity_binding_test"
FILENAME = "D256_L3_H2_attnOnly1_lr5.0e-04_wd0.01.pt"

weights_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [ ]:
REPO_ID = "sojup/entity_binding_test"
FILENAME = "id_to_entity.csv"

id_mapping_path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME)

In [5]:
E = 100
T = 10
D_VOCAB = E + T + 3

In [6]:
N_LAYERS = 3
HEADS = 2

d_model = 256
n_ctx   = 64

def build_model(n_layers: int, n_heads: int) -> HookedTransformer:
    if d_model % n_heads != 0:
        return None
    d_head = d_model // n_heads

    cfg = HookedTransformerConfig(
        n_layers=n_layers,
        n_heads=n_heads,
        d_model=d_model,
        d_head=d_head,
        n_ctx=n_ctx,
        d_vocab=D_VOCAB,
        d_vocab_out=E,
        attn_only=True,
        normalization_type="LN",
    )
    return HookedTransformer(cfg)

model = build_model(N_LAYERS, HEADS)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Load the model
pretrained_weights = torch.load(weights_path, map_location=device, weights_only=True)
state_dict = pretrained_weights["model"]
model.load_state_dict(state_dict)

print("Model loaded successfully.")

Moving model to device:  cuda
Model loaded successfully.


In [7]:
id_mapping_df = pd.read_csv(id_mapping_path)
id_to_entity = dict(zip(id_mapping_df['id'], id_mapping_df['name']))

In [10]:
id_to_entity[100] = 'loves'
id_to_entity[101] = 'works with'
id_to_entity[102] = 'interacts with'
id_to_entity[103] = 'lives with'
id_to_entity[104] = 'has a grudge against'
id_to_entity[105] = 'is interested in'
id_to_entity[106] = 'plays with'
id_to_entity[107] = 'goes to school with'
id_to_entity[108] = 'is jealous of'
id_to_entity[109] = 'wants'


In [11]:
class EntityBindingDataset(Dataset):
    def __init__(self, dataframe, parse_tokens_if_str=True):
        self.df = dataframe.reset_index(drop=True)
        self.parse_tokens_if_str = parse_tokens_if_str

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = row["tokens"]
        tokens = torch.tensor(seq, dtype=torch.long)
        label  = torch.tensor(int(row["label"]), dtype=torch.long)
        return tokens, label

In [ ]:
from datasets import load_dataset

dataset = load_dataset("sojup/entity_binding", split="test")

In [13]:
test_df = dataset.to_pandas()
test_dataset = EntityBindingDataset(test_df)

In [15]:
example, label = test_dataset[0]
example, label, [id_to_entity[i.item()] for i in example], id_to_entity[label.item()]

(tensor([ 10, 104,   1, 110,  20, 103,  30, 110,  70, 105,  37, 110,  30, 100,
          24, 110, 105,  70, 111]),
 tensor(37),
 ['Jeffery',
  'has a grudge against',
  'Angel',
  ',',
  'Linda',
  'lives with',
  'Lindsay',
  ',',
  'Jason',
  'is interested in',
  'Christine',
  ',',
  'Lindsay',
  'loves',
  'Susan',
  ',',
  'is interested in',
  'Jason',
  '?'],
 'Christine')

In [ ]:
## considering the incorrect token to be the 10th most probable prediction
## take a batch of examples with this heuristic

In [16]:
try:
    import google.colab # type: ignore
    IN_COLAB = True
except:
    IN_COLAB = False

import os, sys
chapter = "chapter1_transformer_interp"
repo = "ARENA_3.0"

if IN_COLAB:
    # Install packages
    %pip install einops
    %pip install jaxtyping
    %pip install git+https://github.com/callummcdougall/CircuitsVis.git#subdirectory=python

    # Code to download the necessary files (e.g. solutions, test funcs)
    if not os.path.exists(f"/content/{chapter}"):
        !wget https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/main.zip
        !unzip /content/main.zip 'ARENA_3.0-main/chapter1_transformer_interp/exercises/*'
        sys.path.append(f"/content/{repo}-main/{chapter}/exercises")
        os.remove("/content/main.zip")
        os.rename(f"{repo}-main/{chapter}", chapter)
        os.rmdir(f"{repo}-main")
        os.chdir(f"{chapter}/exercises")
else:
    chapter_dir = r"./" if chapter in os.listdir() else os.getcwd().split(chapter)[0]
    sys.path.append(chapter_dir + f"{chapter}/exercises")


  Cloning https://github.com/callummcdougall/CircuitsVis.git to /tmp/pip-req-build-nj5lva1b
  Running command git clone --filter=blob:none --quiet https://github.com/callummcdougall/CircuitsVis.git /tmp/pip-req-build-nj5lva1b
  Resolved https://github.com/callummcdougall/CircuitsVis.git to commit 1e6129d08cae7af9242d9ab5d3ed322dd44b4dd3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for circuitsvis: filename=circuitsvis-0.0.0-py3-none-any.whl size=6172337 sha256=a4f68a36d13a4fbe2ed1d8bde97d80050bd39c247f48ad5a9f309c1c97601ffd
  Stored in directory: /tmp/pip-ephem-wheel-cache-fdrzajtl/wheels/00/ce/19/651aed367fa8cefad943dece40a2248cef6588697047472ef1
Successfully built circuitsvis
  Attempting uninstall: importlib-metadata
    Found existing installation: importlib_metadata 8.7.0
    Uninstalling importlib_metadata-8.7.0:
      Successfully uninstalled importlib_metadata-8.7.0
--2025-

In [17]:
### Attention Pattern

In [18]:
logits, cache = model.run_with_cache(example, remove_batch_dim=True)

In [35]:
probs = torch.softmax(logits[0, -1, :], dim=-1)

In [43]:
for i, idx in enumerate(torch.topk(probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

Christine 0.9999969005584717
Michelle 9.629898158891592e-07
Patrick 5.643713052450039e-07
Erica 3.5272483955850475e-07
Jeffrey 1.8319308026093495e-07


Correct label: Christine


In [45]:
### Taking Michelle as corrupt ex.

In [46]:
### Method 1: Residual stream patching

In [47]:
example

tensor([ 10, 104,   1, 110,  20, 103,  30, 110,  70, 105,  37, 110,  30, 100,
         24, 110, 105,  70, 111])

In [50]:
id_to_entity_rev = {v: k for k, v in id_to_entity.items()}

In [52]:
id_to_entity_rev["Michelle"], id_to_entity_rev["Christine"]

(99, 37)

In [53]:
corrupt_example = example.clone()
corrupt_example[corrupt_example == id_to_entity_rev["Christine"]] = id_to_entity_rev["Michelle"]

In [54]:
corrupt_example

tensor([ 10, 104,   1, 110,  20, 103,  30, 110,  70, 105,  99, 110,  30, 100,
         24, 110, 105,  70, 111])

In [96]:
corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_example)

In [97]:
corrupt_probs = torch.softmax(corrupt_logits[0, -1, :], dim=-1)

In [98]:
for i, idx in enumerate(torch.topk(corrupt_probs, 5).indices.tolist()):
  print(f"{id_to_entity[idx]} {torch.topk(corrupt_probs, 5).values.tolist()[i]}")

print(f"\n\nCorrect label: {id_to_entity[label.item()]}")

Michelle 0.9999991655349731
Kimberly 2.863485519810638e-07
Nancy 1.5057737812185223e-07
Robert 1.2415841865731636e-07
Allison 9.842160864081961e-08


Correct label: Christine


In [56]:
def patch_residual_stream(activations, hook, layer="blocks.6.hook_resid_post", pos=5):
   activations[:, pos, :] = corrupt_cache[layer][:, pos, :]
   return activations

In [89]:
import torch
from functools import partial

layers = ["blocks.0.hook_resid_pre", *[f"blocks.{i}.hook_resid_post" for i in range(model.cfg.n_layers)]]
n_layers = len(layers)
n_pos = len(example)

clean_answer_index = 37
corrupt_answer_index = 99

# Test the effect of patching at any layer and any position
patching_effect = torch.zeros(n_layers, n_pos)
for l, layer in enumerate(layers):
    for pos in range(n_pos):
        fwd_hooks = [(layer, partial(patch_residual_stream, layer=layer, pos=pos))]
        prediction_logits = model.run_with_hooks(example,
                                                 fwd_hooks=fwd_hooks)[0, -1]
        patching_effect[l, pos] = prediction_logits[clean_answer_index] \
                                  - prediction_logits[corrupt_answer_index]

In [82]:
def imshow(
    tensor,
    xlabel="X",
    ylabel="Y",
    zlabel=None,
    xticks=None,
    yticks=None,
    c_midpoint=0.0,
    c_scale="RdBu",
    show=True,
    **kwargs
):
    tensor = utils.to_numpy(tensor)
    n_rows, n_cols = tensor.shape

    labels = {"x": xlabel, "y": ylabel}
    if zlabel is not None:
        labels["color"] = zlabel

    # Build the figure with numeric axes
    fig = px.imshow(
        tensor,
        labels=labels,
        color_continuous_midpoint=c_midpoint,
        color_continuous_scale=c_scale,
        **kwargs
    )

    # Map numeric positions -> your (possibly duplicate) tokens
    if xticks is not None:
        xtxt = [str(x) for x in xticks]
        fig.update_xaxes(
            tickmode="array",
            tickvals=list(range(n_cols)),
            ticktext=xtxt,
            type="linear"   # ensure numeric axis, not categorical
        )

    if yticks is not None:
        ytxt = [str(y) for y in yticks]
        fig.update_yaxes(
            tickmode="array",
            tickvals=list(range(n_rows)),
            ticktext=ytxt,
            type="linear"
        )

    return fig


In [90]:
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layers, xlabel="pos", ylabel="layer",
       zlabel="Logit difference", title="Patching with other name", width=800, height=380)

In [ ]:
## Hypothesis: Information is stored in Layer 0 at the Christine token (maybe current token attention) - then the information travels to the comma?
## And then just at the last layer, it travels to the final token

In [129]:
def patch_head_result(activations, hook, layer=None, head=None, pos=None):
   activations[:, pos, head, :] = corrupt_cache[hook.name][:, pos, head, :]
   return activations

In [130]:
n_layers = 3
n_heads = 2
n_pos = 19

_, corrupt_cache = model.run_with_cache(corrupt_example)


patching_effect = torch.zeros(n_layers*n_heads, n_pos)
for layer in range(n_layers):
    for head in range(n_heads):
        for pos in range(n_pos):
            fwd_hooks = [(
            	f"blocks.{layer}.attn.hook_result",
	            partial(patch_head_result, layer=layer, head=head, pos=pos)
            )]
            prediction_logits = model.run_with_hooks(example,
                                                     fwd_hooks=fwd_hooks)[0, -1]
            patching_effect[n_heads*layer+head, pos] =  \
                                    prediction_logits[clean_answer_index] \
                                    - prediction_logits[corrupt_answer_index]


token_labels = [f"(pos {i:2}) {t}" for i, t in enumerate(example)]
layerhead_labels = [f"{l}.{h}" for l in range(n_layers) for h in range(n_heads)]
imshow(patching_effect, xticks=[id_to_entity[i.item()] for i in example], yticks=layerhead_labels, xlabel="position", ylabel="layer.head",
           zlabel="Logit difference", title=f"Patching with Michelle instead of Christine", width=700, height=800)

In [131]:
### Attention Pattern

In [44]:
import circuitsvis as cv
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

def tensor_to_numpy(t):
    if isinstance(t, torch.Tensor):
        t = t.detach().cpu().numpy()
    return t

str_tokens = [id_to_entity[i.item()] for i in example]
for layer in range(model.cfg.n_layers):
    attention_pattern = cache["pattern", layer]
    display(Markdown(f"### Layer {layer}"))
    display(cv.attention.attention_patterns(tokens=str_tokens, attention=attention_pattern))

### Layer 0

### Layer 1

### Layer 2

In [22]:
cache["pattern", 0].shape

torch.Size([2, 19, 19])